# Reconstrucción 3D: COLMAP vs VGGT

| | COLMAP | VGGT |
|---|---|---|
| Método | SfM clásico (sparse) | Transformer |
| Tiempo T4 | SIFT + mapper ~6 min | Inferencia ~60 s |
| Superficies lisas | Falla | Funciona |
| Reconstrucción | Sparse → Poisson | Siempre densa |

---

**Cómo ejecutarlo:**

1. Activa GPU T4 — *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4*
2. Ejecuta todo — *Entorno de ejecución → Ejecutar todo* (o `Ctrl+F9`)
3. Cuando llegue la celda de subida, selecciona tus **40-50 fotos del objeto** (JPG o PNG)
4. La última celda descarga automáticamente `colmap_model.obj` y `vggt_model.obj`

Úsalos en local: `python run.py --model=vggt_model.obj`

In [ ]:
!nvidia-smi -L 2>/dev/null || echo "Sin GPU — activa T4: Entorno de ejecucion > Cambiar tipo > T4"

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
!apt-get install -y -q colmap 2>/dev/null
!pip install -q open3d

import os, subprocess
import open3d as o3d
print('Open3D:', o3d.__version__)

In [ ]:
!pip install -q git+https://github.com/facebookresearch/vggt.git huggingface_hub
!pip install -q "numpy>=2.0" --upgrade

In [ ]:
import subprocess, os, time
from pathlib import Path
from google.colab import files

IMG_DIR = Path('images')
IMG_DIR.mkdir(exist_ok=True)

uploaded = files.upload()
for name, data in uploaded.items():
    (IMG_DIR / name).write_bytes(data)

imgs = sorted(p for p in IMG_DIR.iterdir()
              if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'})
print(f'{len(imgs)} imagenes')
assert imgs, 'Sube al menos 5 imagenes'

---
## COLMAP — Structure from Motion (sparse)

In [ ]:
DB = 'colmap.db'
os.makedirs('sparse', exist_ok=True)

t0 = time.time()

subprocess.run(['colmap', 'feature_extractor',
    '--database_path', DB, '--image_path', str(IMG_DIR),
    '--ImageReader.single_camera', '1',
    '--SiftExtraction.use_gpu', '0'], check=True)

subprocess.run(['colmap', 'exhaustive_matcher',
    '--database_path', DB,
    '--SiftMatching.use_gpu', '0'], check=True)

t_feat = time.time() - t0
print(f'Extraccion + matching: {t_feat:.0f}s')

In [ ]:
t0 = time.time()

r = subprocess.run(['colmap', 'mapper',
    '--database_path', DB, '--image_path', str(IMG_DIR),
    '--output_path', 'sparse'], capture_output=True, text=True)

if not os.path.exists('sparse/0'):
    print(r.stdout[-3000:])
    raise RuntimeError('Mapper fallo — revisa solapamiento/calidad de las fotos')

os.makedirs('sparse_txt', exist_ok=True)
subprocess.run(['colmap', 'model_converter',
    '--input_path', 'sparse/0', '--output_path', 'sparse_txt',
    '--output_type', 'TXT'], check=True, capture_output=True)

t_map = time.time() - t0
n_cam = sum(1 for l in open('sparse_txt/cameras.txt')  if l[0] != '#' and l.strip())
n_pts = sum(1 for l in open('sparse_txt/points3D.txt') if l[0] != '#' and l.strip())

colmap_stats = {'t_feat': t_feat, 't_map': t_map, 'n_pts': n_pts, 'n_cam': n_cam}
print(f'Mapper: {t_map:.0f}s | {n_cam}/{len(imgs)} camaras | {n_pts} puntos 3D')

In [ ]:
import numpy as np, open3d as o3d


def read_points3d(path):
    pts, cols = [], []
    with open(path) as fh:
        for line in fh:
            if line.startswith('#') or not line.strip():
                continue
            t = line.split()
            pts.append([float(t[1]), float(t[2]), float(t[3])])
            cols.append([int(t[4])/255., int(t[5])/255., int(t[6])/255.])
    return np.array(pts, np.float64), np.array(cols, np.float64)


DENSE_PLY = 'dense/fused.ply'

if os.path.exists(DENSE_PLY):
    print('Usando reconstruccion densa')
    pcd_c = o3d.io.read_point_cloud(DENSE_PLY)
    depth = 9
else:
    print('Usando reconstruccion sparse')
    pts_c, cols_c = read_points3d('sparse_txt/points3D.txt')
    pcd_c = o3d.geometry.PointCloud()
    pcd_c.points = o3d.utility.Vector3dVector(pts_c)
    pcd_c.colors = o3d.utility.Vector3dVector(cols_c)
    depth = 8

print(f'{len(pcd_c.points):,} puntos')
pcd_c, _ = pcd_c.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
pcd_c.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
pcd_c.orient_normals_consistent_tangent_plane(30)

t0 = time.time()
mesh_c, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_c, depth=depth)
mesh_c.remove_vertices_by_mask(np.asarray(dens) < np.quantile(np.asarray(dens), 0.15))
mesh_c = mesh_c.filter_smooth_simple(5)
o3d.io.write_triangle_mesh('colmap_model.obj', mesh_c)

colmap_stats['n_tris'] = len(mesh_c.triangles)
colmap_stats['t_mesh'] = time.time() - t0
colmap_stats['dense']  = os.path.exists(DENSE_PLY)
print(f'{len(mesh_c.vertices)} verts, {len(mesh_c.triangles)} tris → colmap_model.obj  ({colmap_stats["t_mesh"]:.0f}s)')

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files as _colab_files

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d')
_p = np.asarray(pcd_c.points)
if len(_p) > 2000:
    _p = _p[np.random.default_rng(0).choice(len(_p), 2000, replace=False)]
ax.scatter(_p[:,0], _p[:,1], _p[:,2], s=0.5, c=_p[:,2], cmap='viridis', alpha=0.7)
ax.set_title(f'COLMAP  {colmap_stats["n_pts"]:,} pts  ·  {colmap_stats["n_tris"]:,} tris')
ax.set_axis_off()
plt.tight_layout()
plt.show()

_colab_files.download('colmap_model.obj')

---
## VGGT

Modelo `facebook/vggt-1B` (~4 GB, descarga la primera vez). Si hay OOM reduce las imágenes a 20-25.

In [ ]:
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32

t0 = time.time()
model = VGGT.from_pretrained('facebook/vggt-1B').to(device).eval()
print(f'Modelo: {time.time()-t0:.0f}s')

t0 = time.time()
frames = load_and_preprocess_images([str(p) for p in imgs]).to(device).to(dtype)
with torch.no_grad():
    preds = model(frames.unsqueeze(0))

world_pts  = preds['world_points'][0].cpu().float().numpy()
world_conf = preds['world_points_conf'][0].cpu().float().numpy()
del model, frames, preds
torch.cuda.empty_cache()

t_vggt = time.time() - t0
print(f'Inferencia: {t_vggt:.0f}s')

In [ ]:
CONF_THRESH = 0.95   # sube desde 0.5 para eliminar ruido exterior
MAX_PTS     = 400_000

mask  = world_conf > CONF_THRESH
pts_v = world_pts[mask].reshape(-1, 3)
print(f'Puntos (conf>{CONF_THRESH}): {len(pts_v):,}')

if len(pts_v) > MAX_PTS:
    idx   = np.random.default_rng(0).choice(len(pts_v), MAX_PTS, replace=False)
    pts_v = pts_v[idx]

pts_v = pts_v[np.isfinite(pts_v).all(axis=1)]  # eliminar NaN/inf
pts_v = np.ascontiguousarray(pts_v.astype(np.float64))

pcd_v = o3d.geometry.PointCloud()
pcd_v.points = o3d.utility.Vector3dVector(pts_v)
pcd_v, _ = pcd_v.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
pcd_v.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
pcd_v.orient_normals_consistent_tangent_plane(30)

t0 = time.time()
mesh_v, dens = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd_v, depth=9)
mesh_v.remove_vertices_by_mask(np.asarray(dens) < np.quantile(np.asarray(dens), 0.10))
mesh_v = mesh_v.filter_smooth_simple(3)
o3d.io.write_triangle_mesh('vggt_model.obj', mesh_v)

vggt_stats = {'t_infer': t_vggt, 't_mesh': time.time()-t0,
              'n_pts': len(pts_v), 'n_tris': len(mesh_v.triangles)}
print(f'{len(mesh_v.vertices)} verts, {len(mesh_v.triangles)} tris → vggt_model.obj  ({vggt_stats["t_mesh"]:.0f}s)')

In [ ]:
import matplotlib.pyplot as plt
from google.colab import files as _colab_files

fig = plt.figure(figsize=(7, 5))
ax = fig.add_subplot(111, projection='3d')
_p = np.asarray(pcd_v.points)
if len(_p) > 2000:
    _p = _p[np.random.default_rng(0).choice(len(_p), 2000, replace=False)]
ax.scatter(_p[:,0], _p[:,1], _p[:,2], s=0.5, c=_p[:,2], cmap='plasma', alpha=0.7)
ax.set_title(f'VGGT  {vggt_stats["n_pts"]:,} pts  ·  {vggt_stats["n_tris"]:,} tris')
ax.set_axis_off()
plt.tight_layout()
plt.show()

_colab_files.download('vggt_model.obj')

---
## Comparación COLMAP vs VGGT

In [ ]:
t_c = colmap_stats['t_feat'] + colmap_stats['t_map'] + colmap_stats['t_mesh']
t_v = vggt_stats['t_infer'] + vggt_stats['t_mesh']

print(f'{"":<30} {"COLMAP":>12} {"VGGT":>12}')
print('-' * 54)
print(f'{"Tiempo total (s)":<30} {t_c:>12.0f} {t_v:>12.0f}')
print(f'{"Puntos 3D":<30} {colmap_stats["n_pts"]:>12,} {vggt_stats["n_pts"]:>12,}')
print(f'{"Triangulos":<30} {colmap_stats["n_tris"]:>12,} {vggt_stats["n_tris"]:>12,}')
print(f'{"Camaras registradas":<30} {colmap_stats["n_cam"]:>12} {len(imgs):>12}')
print(f'{"Reconstruccion densa":<30} {"si" if colmap_stats.get("dense") else "no":>12} {"siempre":>12}')

In [ ]:
import matplotlib.pyplot as plt

def sample(pcd, n=2000):
    pts = np.asarray(pcd.points)
    if len(pts) > n:
        idx = np.random.default_rng(0).choice(len(pts), n, replace=False)
        pts = pts[idx]
    return pts

fig = plt.figure(figsize=(13, 5))
for k, (pcd, title) in enumerate([
    (pcd_c, f'COLMAP  {colmap_stats["n_pts"]:,} pts'),
    (pcd_v, f'VGGT  {vggt_stats["n_pts"]:,} pts'),
]):
    ax = fig.add_subplot(1, 2, k+1, projection='3d')
    p = sample(pcd)
    ax.scatter(p[:,0], p[:,1], p[:,2], s=0.5, c=p[:,2], cmap='viridis', alpha=0.7)
    ax.set_title(title)
    ax.set_axis_off()
plt.tight_layout()
plt.savefig('comparacion.png', dpi=120)
plt.show()

In [ ]:
import plotly.graph_objects as go

def _mesh3d(mesh, name):
    v = np.asarray(mesh.vertices)
    t = np.asarray(mesh.triangles)
    return go.Mesh3d(x=v[:,0], y=v[:,1], z=v[:,2],
                     i=t[:,0], j=t[:,1], k=t[:,2],
                     opacity=0.6, name=name)

go.Figure([_mesh3d(mesh_c, 'COLMAP'), _mesh3d(mesh_v, 'VGGT')],
          layout=dict(scene_aspectmode='data')).show()

In [ ]:
# Re-descarga por si algún archivo no se descargó automáticamente antes
from google.colab import files
for f in ['colmap_model.obj', 'vggt_model.obj', 'comparacion.png']:
    if os.path.exists(f):
        files.download(f)